<a href="https://colab.research.google.com/github/jollyoli93/jackoliver_24862664_dissertation/blob/main/Crane_Detector_RegressionModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Startup

In [ ]:
!git clone https://github.com/jollyoli93/jackoliver_24862664_dissertation

In [ ]:
!unzip /content/jackoliver_24862664_dissertation/train_labels.zip -d /content/
!unzip /content/jackoliver_24862664_dissertation/val_labels.zip -d /content/

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:

import numpy as np
import torch
import itertools
import pandas as pd
import matplotlib.pyplot as plt

## Regression Algo

In [ ]:
from pathlib import Path
from pandas import DataFrame

In [ ]:
train_labels = "/content/labels"
valid_labels = "/content/val_labels"

In [ ]:
train_coords = [p for p in Path(train_labels).glob("*")]
valid_coords = [p for p in Path(valid_labels).glob("*")]

In [ ]:
len(train_coords), len(valid_coords)

In [ ]:
print(coords[2])

In [ ]:
def get_data_dist(path):
  coords = [p for p in Path(path).glob("*")]
  background = 0
  coord_split_list = []
  for coord in coords:
    with open(coord, "r") as f:
      coord = f.readlines()
      if len(coord) == 0:
        background += 1

      coord_split = [
        [float(x) for x in line.strip().split()]
        for line in coord
      if line.strip()
      ]

      coord_split_list.extend(coord_split)
  print(coord_split_list)
  hooks = [d for d in coord_split_list if d[0] == 0]  # hook
  loads = [d for d in coord_split_list if d[0] == 1]  # load

  return (len(hooks), len(loads), background)

In [ ]:
train_dist = get_data_dist(train_labels)
valid_dist = get_data_dist(valid_labels)

In [ ]:
train_dist, valid_dist

In [ ]:
h, l, b = train_dist
vh, vl, vb = valid_dist

In [ ]:
# print(f"Train Hooks: {h}, Train Loads: {l}")
# print(f"Valid Hooks: {vh}, Valid Loads: {vl}")

th = h+vh
tl = l + vl
tb = b + vb

total = th + tl + tb
percent_h = int((th/total)*100)
percent_l = int((tl/total)*100)
percent_b = int((tb/total)*100)

print(f"Total Hooks: {th}, Total Loads: {tl}, Total Background {tb}")
print(f"Total Instances: {total}")
print(f"Hook %: {percent_h}, Load % {percent_l}, Background % {percent_b}")

### Get Pairs

In [ ]:
yolo_txt = [1, 0.48534666666666665, 0.2776875, 0.005231666666666667, 0.013035]


In [ ]:
from math import sqrt

def dist_between_centroids(input, target):
  x1, y1 = input
  x2, y2 = target

  dx = x1 - x2
  dy = y1 - y2

  distance = sqrt(dx**2 + dy**2)
  return distance

In [ ]:
dist_between_centroids(yolo_txt[1:3], (0.4, 0.16))

In [ ]:
# def find_distance(input, target):
#   input_centroid = find_centroid(input)
#   target_centroid = find_centroid(target)

#   distance = dist_between_centroids(input_centroid, target_centroid)
#   return distance


In [ ]:
from math import sqrt

def get_pairs(coords):
    coord_split = [
        [float(x) for x in line.strip().split()]
        for line in coords
        if line.strip()
    ]

    hooks = [d for d in coord_split if d[0] == 0]  # hook
    loads = [d for d in coord_split if d[0] == 1]  # load

    print(hooks)
    print(loads)
    pairs = {'hook': [], 'load': []}
    used_hooks = set()

    for load in loads:
        xc_load, yc_load = load[1], load[2]
        h_load = load[-1]
        closest = None
        closest_idx = -1
        min_dist = float("inf")

        for i, hook in enumerate(hooks):
            if i in used_hooks:
                continue

            xc_hook, yc_hook = hook[1], hook[2]

            yt_load = yc_load - (h_load /2)
            yb_hook = yc_hook + hook[-1]/2 #Location at bottom of hook

            # Hook must be above load
            if yb_hook > yt_load:
                continue

            # Horizontal closeness
            dist = abs(xc_load - xc_hook)
            if dist < min_dist:
                min_dist = dist
                closest = hook
                closest_idx = i

        if closest is not None:
            xc, yc, w, h = closest[1:5]
            yb = yc + (h / 2) # positive number is down in cv2 gridspace

            # get relative offsets by normalising the difference by the hooks dimension. So offset is relative to the hook rather than the image
            dx = (xc_load - xc) /w
            dy = (yc_load - yb) /h

            pairs['hook'].append(
                [xc, yc, w, h] + [yb]
            )
            pairs['load'].append([dx, dy])

            used_hooks.add(closest_idx)

    return pairs


## Create Dataset

In [ ]:
from torch.utils.data import Dataset

class YoloLabelsDataset(Dataset):
    def __init__(self, label_dir):
        self.label_dir = label_dir
        self.labels = list(Path(label_dir).glob("*.txt"))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        label = self.labels[idx]

        with open(label, "r") as f:
          coord_file = f.readlines()

        pairs = get_pairs(coord_file)

        if (pairs is None or
            'hook' not in pairs or 'load' not in pairs or
            len(pairs['hook']) == 0 or len(pairs['load']) == 0):
            return None
        return pairs

In [ ]:
train_ds = YoloLabelsDataset(train_labels)
valid_ds = YoloLabelsDataset(valid_labels)

In [ ]:
len(train_ds), len(valid_ds)

In [ ]:
train_ds.labels[2]

In [ ]:
test_data = train_ds.__getitem__(2)

In [ ]:
batch = next(iter(train_ds))

In [ ]:
test_valid_data = valid_ds.__getitem__(6)

## Get scalars for normalising the offsets


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Collect all data for fitting
train_feats = []
train_targets = []

for label_file in Path(train_labels).glob("*.txt"):
    with open(label_file) as f:
        pairs = get_pairs(f.readlines())
        if pairs:
            train_feats.extend(pairs['hook'])
            train_targets.extend(pairs['load'])

# Fit the scalers
feat_scaler = StandardScaler().fit(train_feats)
target_scaler = StandardScaler().fit(train_targets)

In [ ]:
feat_scaler.mean_, target_scaler.mean_

## Collate Function

In [ ]:
def collate_pairs(batch, feat_scale, target_scale):
    batch = [item for item in batch if item is not None]

    xs = []
    ys = []

    for item in batch:
        for x_coords, y_coords in zip(item['hook'], item['load']):
            if(len(x_coords)==0 or len(y_coords)==0):
              continue

            xs.append(x_coords)
            ys.append(y_coords)

    if len(xs) == 0:
      return None

    xs_scaled = feat_scale.transform(xs)
    ys_scaled = target_scale.transform(ys)

    x_tens = torch.tensor(xs_scaled, dtype=torch.float32)
    y_tens = torch.tensor(ys_scaled, dtype=torch.float32)

    return x_tens, y_tens



In [ ]:
collate_pairs([test_data], feat_scaler, target_scaler)

## Create Dataloaders

In [ ]:
torch.manual_seed(42)

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_ds, batch_size=400, collate_fn=lambda b: collate_pairs(b, feat_scaler, target_scaler), shuffle=True)
valid_dataloader = DataLoader(valid_ds, batch_size=80, collate_fn=lambda b: collate_pairs(b, feat_scaler, target_scaler), shuffle=True)

In [ ]:
batch = next(iter(train_dataloader))

In [ ]:
x, y = batch

In [ ]:
x.shape, y.shape

In [ ]:
# batch = next(iter(valid_dataloader))
# x, y = batch
# x.shape, y.shape

In [ ]:
valid_batch = next(iter(train_dataloader))

# Define Test Model

In [ ]:
class LinearRegression(torch.nn.Module):
    def __init__(self, in_feat, out_feat) -> None:
        super().__init__()
        self.linear = torch.nn.Sequential(
            torch.nn.Linear(in_feat, 100),
            torch.nn.ReLU(),
            torch.nn.Linear(100, 100),
            torch.nn.ReLU(),
            torch.nn.Linear(100, out_feat),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        x = self.linear(x)
        return x

linearModel = LinearRegression(5, 2)

optimizer = torch.optim.SGD(linearModel.parameters(), lr=0.1, momentum=0.9)
loss = torch.nn.MSELoss()

In [ ]:
linearModel = LinearRegression(5, 2)

In [ ]:
optimizer = torch.optim.SGD(linearModel.parameters(), lr=0.1, momentum=0.9)
loss = torch.nn.MSELoss()

In [ ]:
def train_epoch(dataloader, optim, loss_fn, model):
  for i, batch in enumerate(dataloader):
    x,y = batch

    # add jitter to the model
    if model.training:
        x = x + (torch.randn_like(x) * 0.005)
        print(x.shape)
    predict = model(x)
    loss = loss_fn(predict, y)
    optim.zero_grad()
    loss.backward()
    optim.step()
  return loss.item()

In [ ]:
# train_epoch(train_dataloader, optimizer, loss, linearModel)

In [ ]:
# len(valid_dataloader.dataset)

In [ ]:
from sklearn.metrics import r2_score

@torch.no_grad()
def validate_epoch(dataloader, model, loss_fn):
  total_loss = 0
  all_targets = []
  all_preds = []

  for i, batch in enumerate(dataloader):
    x, y = batch
    predict = model(x)
    print(f"y {y[0]} - predict {predict[0]}")
    loss = loss_fn(predict, y)
    total_loss += loss.item()

    all_preds.append(predict.cpu())
    all_targets.append(y.cpu())

  all_preds = torch.cat(all_preds, dim=0).numpy()
  all_targets = torch.cat(all_targets, dim=0).numpy()

  # Compute r2
  r2 = r2_score(all_targets, all_preds)
  avg_loss = total_loss / len(dataloader.dataset)

  return avg_loss, r2


In [ ]:
# validate_epoch(valid_dataloader, linearModel, loss)

In [ ]:
def fit(epochs):
  train_losses = []
  valid_losses = []
  rsqrd = []

  for i in range(1, epochs+1):
    train_loss = train_epoch(train_dataloader, optimizer, loss, linearModel)
    valid_loss = validate_epoch(valid_dataloader, linearModel, loss)
    mse, r2 = valid_loss

    train_losses.append(train_loss)
    valid_losses.append(mse)
    rsqrd.append(r2)

    print(f"Epoch {i}- Train Loss: {train_loss} - Valid Loss: {mse} -  R2 Loss: {r2}")
  return train_losses, valid_losses, rsqrd

In [ ]:
epochs =10

metrics = fit(epochs)

In [ ]:
metrics

### Plot graphs


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
train_losses, valid_losses, rsqrd = metrics

In [ ]:
plt.plot(range(1, epochs+1), train_losses)
plt.plot(range(1, epochs+1), valid_losses)
plt.show()
plt.plot(range(1, epochs+1), rsqrd)
plt.show()

In [ ]:
rsqrd

In [ ]:
fig, ax = plt.subplots()
ax.set_title("R\u00b2 across epochs")
# ax.grid(True)
ax.set_ylabel("R\u00b2 Score")
ax.set_xlabel("Epoch")
ax.plot(range(1, 11), rsqrd, label="line")
ax.legend()

# Experimental Training

## Define Training Loops

In [ ]:
class LinearRegression(torch.nn.Module):
    def __init__(self, in_feat, out_feat, hidden) -> None:
        super().__init__()
        self.linear = torch.nn.Sequential(
            torch.nn.Linear(in_feat, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, out_feat),
        )

    def forward(self, x):
        x = self.linear(x)
        return x

In [ ]:
import torch
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

class CoordCropModel:
    def __init__(self, model, opt_class, loss, dataloaders, lr=0.1, device=None, momentum=0.9, debug=False) -> None:
        self.model = model
        self.loss_fn = loss
        self.train_dl = dataloaders['train']
        self.valid_dl = dataloaders['valid']
        self.lr = lr
        self.momentum = momentum
        self.debug = debug

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        if opt_class == torch.optim.Adam:
          self.opt = opt_class(self.model.parameters(), lr=self.lr)
        else:
            self.opt = torch.optim.SGD(self.model.parameters(), lr=self.lr, momentum=self.momentum)

        self.train_loss = []
        self.valid_loss = []
        self.r2 = []
        self.epochs = 0

    def train_epoch(self):
        self.model.train()
        epoch_loss = 0.0
        batches_process = 0

        for data in self.train_dl:
          if data is None:
            print("No data")
            continue
          x, y, = data

          if self.model.training:
            x = x + (torch.randn_like(x) * 0.005)

          x, y = x.to(self.device), y.to(self.device)

          predict = self.model(x)
          loss = self.loss_fn(predict, y)

          self.opt.zero_grad()
          loss.backward()

          #DEBUG - Check weights
          if self.debug == True:
            for name, param in self.model.named_parameters():
              if param.grad is not None:
                  print(f"debug {name} grad norm:", param.grad.abs().sum().item())
              else:
                print("Grads are blank")

          self.opt.step()
          epoch_loss += loss.item()

          batches_process += 1

        if batches_process > 0:
          self.train_loss.append(epoch_loss / batches_process if batches_process > 0 else 0)

    @torch.no_grad()
    def validate_epoch(self):
        self.model.eval()
        total_loss = 0.0
        all_targets, all_preds = [], []
        batches_process = 0

        for data in self.valid_dl:
          if data is None:
            print("No data")
            continue
          x, y = data
          x, y = x.to(self.device), y.to(self.device)
          predict = self.model(x)
          loss = self.loss_fn(predict, y)
          total_loss += loss.item()

          all_preds.append(predict.cpu())
          all_targets.append(y.cpu())
          batches_process += 1

        all_preds = torch.cat(all_preds, dim=0).numpy()
        all_targets = torch.cat(all_targets, dim=0).numpy()

        ## Prevent NaN/Inf errors when inversing transform
        if np.isnan(all_preds).any() or np.isinf(all_preds).any():
            print("Warning: NaNs or Infs detected in predictions!")
            self.r2.append(-999.0)
            self.valid_loss.append(999.0)
            return

        # Convert back to original values
        preds = target_scaler.inverse_transform(all_preds)
        targs = target_scaler.inverse_transform(all_targets)
        r2 = r2_score(targs, preds)


        self.r2.append(r2)
        self.valid_loss.append(total_loss / batches_process if batches_process > 0 else 0)

    def fit(self, epochs, print_metrics=True):
      self.epochs += epochs

      for i in range(1, epochs+1):
          self.train_epoch()
          self.validate_epoch()

          if print_metrics:
            print(f"Epoch {i} - "
                  f"Train Loss: {self.train_loss[-1]:.4f} - "
                  f"Valid Loss: {self.valid_loss[-1]:.4f} - "
                  f"R2: {self.r2[-1]:.4f}")

    def get_loss_graph(self, title="", legend="upper right"):
      fig, ax = plt.subplots()
      ax.set_title(f"Loss across epochs {title}")

      ax.set_ylabel("Loss")
      ax.set_xlabel("Epoch")

      (p1,) = ax.plot(range(1, self.epochs+1), self.train_loss, linestyle="-", label="Training Loss")
      (p2,) = ax.plot(range(1, self.epochs+1), self.valid_loss, linestyle="--", label="Validation Loss")
      ax.legend(loc=legend)

    def get_r2_graph(self, title=""):
      fig, ax = plt.subplots()
      ax.set_title(f"R\u00b2 across epochs {title}")
      ax.set_ylabel("R\u00b2 Score")
      ax.set_xlabel("Epoch")
      ax.plot(range(1, self.epochs+1), self.r2)

    def metrics(self):
      metrics = self.train_loss, self.valid_loss, self.r2
      return metrics

In [ ]:
len(train_ds), len(valid_ds)

In [ ]:
from torch.utils.data import DataLoader

seed = 42

g = torch.Generator()
g.manual_seed(seed)

train_dataloader = DataLoader(
    train_ds,
    batch_size=len(train_ds),
    collate_fn=lambda b: collate_pairs(b, feat_scaler, target_scaler),
    shuffle=True,
    generator=g,
    drop_last=True
)

valid_dataloader = DataLoader(
    valid_ds,
    batch_size=len(train_ds),
    collate_fn=lambda b: collate_pairs(b, feat_scaler, target_scaler),
    shuffle=False,
    generator=g,
    drop_last=False
)

In [ ]:
dataloader_dict = {'train': train_dataloader, 'valid': valid_dataloader}

## Test Training Loop

In [ ]:
batch = dataloader_dict['train']

In [ ]:
loss = torch.nn.MSELoss()
opt = torch.optim.SGD
LinearBasic = LinearRegression(5, 2, 100) # 5 features output x,y

cropModel1 = CoordCropModel(model=LinearBasic, loss=loss, opt_class=opt, dataloaders=dataloader_dict, lr=0.0005)

In [ ]:
cropModel1.fit(5)

In [ ]:
cropModel1.get_r2_graph("for cropModel1")

In [ ]:
cropModel1.get_loss_graph("for cropModel1", legend="center right")


In [ ]:
huber_loss = torch.nn.HuberLoss()
opt = torch.optim.Adam
Linear64 = LinearRegression(5, 2, 100) # 5 features output x,y

cropModel1 = CoordCropModel(model=Linear64, loss=huber_loss, opt_class=opt, dataloaders=dataloader_dict, lr=0.0005)

## Grid Search through parameters

### SGD

In [ ]:
param_grid = {
    'lr': [0.1, 0.01, 0.001],
    'momentum' : [0.0, 0.5, 0.9],
    'loss' : [torch.nn.MSELoss(), torch.nn.HuberLoss()],
    'hidden_layer':[10, 50, 100, 200]
}

In [ ]:
keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

records = []

for i, params in enumerate(experiments):
    print(f"\n--- Running Trial {i+1}/{len(experiments)}: {params} ---")

    model = LinearRegression(5, 2, params['hidden_layer'])

    runner = CoordCropModel(
        model=model,
        loss=params['loss'],
        dataloaders=dataloader_dict,
        lr=params['lr'],
        opt_class='SGD',
        momentum=params['momentum']
    )

    try:
      runner.fit(15, print_metrics=False)

      records.append({
          **params,
          'opt_name': 'SGD',
          'final_train_loss': runner.train_loss[-1],
          'final_valid_loss': runner.valid_loss[-1],
          'final_r2': runner.r2[-1],
          'best_r2': max(runner.r2)
      })

    except Exception as e:
            print(f"Trial {i+1} failed at training with error: {e}")
            continue

In [ ]:
results_df = pd.DataFrame(records)

In [ ]:
results_df

In [ ]:
results_df.sort_values(by='final_r2', ascending=False)

In [ ]:
best_trial = results_df.sort_values(by='final_r2', ascending=False).iloc[0]

In [ ]:
best_trial

In [ ]:
results_df.to_csv("/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/GridSearch/sgd_01")

#### Run Best SGD

In [ ]:
best = records[19]

In [ ]:
best['hidden_layer']

In [ ]:
bestSGD = LinearRegression(5, 2, 200) # 5 features output x,y

bestSgdModel = CoordCropModel(model=bestSGD, loss=torch.nn.MSELoss(), opt_class='SGD', dataloaders=dataloader_dict, lr=0.1)

In [ ]:
bestSgdModel.fit(100)

In [ ]:
bestSgdModel.get_loss_graph("SGD-MSE"), bestSgdModel.get_r2_graph("SGD-MSE")

#### Run Best Huber SGD

In [ ]:
huber100Sgd = LinearRegression(5, 2, 200)

huber100SgdModel = CoordCropModel(
    model=huber100Sgd,
    loss=torch.nn.HuberLoss(),
    opt_class='SGD',
    dataloaders=dataloader_dict,
    lr=0.1
    )

huber100SgdModel.fit(100)

In [ ]:
huber100SgdModel.get_loss_graph("SGD-Huber"), huber100SgdModel.get_r2_graph("SGD-Huber")

In [ ]:
trn1, valid1, r21 = bestSgdModel.metrics()
trn2, valid2, r22 = huber100SgdModel.metrics()

In [ ]:
sdg = {"train":trn1, "valid":valid1, "r2":r21}

In [ ]:
sdgMse = pd.DataFrame(sdg); sdgMse.to_csv("/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/results/sdgMse.csv")

In [ ]:
sgdH = {"train":trn2, "valid":valid2, "r2":r22}
sdgHuber = pd.DataFrame(sgdH)
sdgHuber.to_csv("/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/results/sdgHuber.csv")

### Adam

In [ ]:
param_grid = {
    'lr': [0.01, 0.005, 0.001], #crashing at 0.1 and 0.0005
    'loss' : [torch.nn.MSELoss(), torch.nn.HuberLoss()],
    'opt_class': [torch.optim.Adam],
    'hidden_layer':[50, 100, 200, 250]
}

In [ ]:
keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

records = []

for i, params in enumerate(experiments):
    print(f"\n--- Running Trial {i+1}/{len(experiments)}: {params} ---")

    model = LinearRegression(5, 2, params['hidden_layer'])

    runner = CoordCropModel(
        model=model,
        loss=params['loss'],
        dataloaders=dataloader_dict,
        lr=params['lr'],
        opt_class=params['opt_class']
    )

    try:
      runner.fit(20, print_metrics=True) #increase epochs for dur to lower LRs

      records.append({
          **params,
          'opt_name': runner.opt,
          'final_train_loss': runner.train_loss[-1],
          'final_valid_loss': runner.valid_loss[-1],
          'final_r2': runner.r2[-1],
          'best_r2': max(runner.r2)
      })

    except Exception as e:
            print(f"Trial {i+1} failed at training with error: {e}")
            continue

In [ ]:
results_df_adam = pd.DataFrame(records)

In [ ]:
results_df_adam

In [ ]:
results_df_adam.to_csv("/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/GridSearch/adam_01")

In [ ]:
best_trial_adam = results_df_adam.sort_values(by='final_r2', ascending=False); best_trial_adam

In [ ]:
best_trial_adam = results_df_adam.sort_values(by='final_r2', ascending=False).iloc[0:3]; best_trial_adam

In [ ]:
best_trial_adam_huber = (
    results_df_adam[results_df_adam['loss'].astype(str).str.contains('HuberLoss')]
    .sort_values(by='final_r2', ascending=False)
    .iloc[0:3]
)

best_trial_adam_huber

#### Run best Adam (MSE)

In [ ]:
mse100 = LinearRegression(5, 2, 100) # 5 features output x,y

mse100Model = CoordCropModel(
    model=mse100,
    loss=torch.nn.MSELoss(),
    opt_class=torch.optim.Adam,
    dataloaders=dataloader_dict,
    lr=0.01
    )
mse100Model.fit(100)

In [ ]:
mse100Model.get_loss_graph("for Adam-MSE"), mse100Model.get_r2_graph("for Adam-MSE")

In [ ]:
trn1, valid1, r21 = mse100Model.metrics()

metrics = {"train":trn1, "valid":valid1, "r2":r21}
metricsDF = pd.DataFrame(metrics)
metricsDF.to_csv("/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/results/AdamMSE.csv")

#### Run best Adam (HuberLoss)

In [ ]:
adam250 = LinearRegression(5, 2, 250) # 5 features output x,y

adam250Model = CoordCropModel(
    model=adam250,
    loss=torch.nn.HuberLoss(),
    opt_class=torch.optim.Adam,
    dataloaders=dataloader_dict,
    lr=0.005
    )
adam250Model.fit(100)

In [ ]:
adam250Model.get_loss_graph("for Adam-Huber")

In [ ]:
adam250Model.get_r2_graph("for Adam-Huber")

In [ ]:
trn1, valid1, r21 = adam250Model.metrics()

metrics = {"train":trn1, "valid":valid1, "r2":r21}
metricsDF = pd.DataFrame(metrics)
metricsDF.to_csv("/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/results/AdamHuber.csv")

### Plot experiments

In [ ]:
SgdMSE = pd.read_csv("/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/results/sdgMse.csv")
SgdHuber = pd.read_csv("/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/results/sdgHuber.csv")
AdamMSE = pd.read_csv("/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/results/AdamMSE.csv")
AdamHuber = pd.read_csv("/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/results/AdamHuber.csv")

In [ ]:
(v1, v2, v3) = SgdMSE[cols].values.squeeze()

In [ ]:
t1, v1, r1 = SgdMSE['train'].to_numpy(), SgdMSE['valid'].to_numpy(), SgdMSE['r2'].to_numpy()
t2, v2, r2 = SgdHuber['train'].to_numpy(), SgdHuber['valid'].to_numpy(), SgdHuber['r2'].to_numpy()
t3, v3, r3 = AdamMSE['train'].to_numpy(), AdamMSE['valid'].to_numpy(), AdamMSE['r2'].to_numpy()
t4, v4, r4 = AdamHuber['train'].to_numpy(), AdamHuber['valid'].to_numpy(), AdamHuber['r2'].to_numpy()


In [ ]:
r2s= [r1,r2,r3,r4]

for i, r in enumerate(r2s):
    max_val = round(r.max(), 3)
    best_epoch = r.argmax() + 1

    print(max_val, best_epoch)

In [ ]:
fig, ax = plt.subplots()
ax.set_title("Best R\u00b2 across epochs")
# ax.grid(True)
ax.set_ylabel("R\u00b2 Score")
ax.set_xlabel("Epoch")
ax.plot(range(1, 101), r1, label="SGD-MSE-200")
ax.plot(range(1, 101), r2, label="SGD-Huber-100")
ax.plot(range(1, 101), r3, label="Adam-MSE-100")
ax.plot(range(1, 101), r4, label="Adam-Huber-250")
ax.legend()

In [ ]:
fig, ax = plt.subplots()
ax.set_title("Loss across epochs")
# ax.grid(True)
ax.set_ylabel("R\u00b2 Score")
ax.set_xlabel("Epoch")
ax.plot(range(1, 101), t1, label="SGD-MSE-200")
ax.plot(range(1, 101), t2, label="SGD-Huber-100")
ax.plot(range(1, 101), t3, label="Adam-MSE-100")
ax.plot(range(1, 101), t4, label="Adam-Huber-250")
ax.legend()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

style_handles = [
    Line2D([0], [0], color="gray", linestyle="-", label="Train"),
    Line2D([0], [0], color="gray", linestyle="--", label="Validation"),
]

# MSE
ax2.set_title("MSE Loss across epochs")
ax2.set_ylabel("Loss")
ax2.set_xlabel("Epoch")

(p3,) = ax2.plot(range(1, 101), t1, linestyle="-", label="SGD-MSE-200")
ax2.plot(range(1, 101), v1, linestyle="--", color=p3.get_color())
(p4,) = ax2.plot(range(1, 101), t3, linestyle="-", label="Adam-MSE-100")
ax2.plot(range(1, 101), v3, linestyle="--", color=p4.get_color())

ax2.legend(handles=[p3, p4] + style_handles, loc="center right")

# Huber
ax1.set_title("Huber Loss across epochs")
ax1.set_ylabel("Loss")
ax1.set_xlabel("Epoch")

(p1,) = ax1.plot(range(1, 101), t2, linestyle="-", label="SGD-Huber-100")
ax1.plot(range(1, 101), v2, linestyle="--", color=p1.get_color())
(p2,) = ax1.plot(range(1, 101), t4, linestyle="-", label="Adam-Huber-250")
ax1.plot(range(1, 101), v4, linestyle="--", color=p2.get_color())

ax1.legend(handles=[p1, p2] + style_handles, loc="upper right")

plt.tight_layout()
plt.show()

### Test run

#### Adam50 with LR 0.01

In [ ]:
adam50 = LinearRegression(5, 2, 50) # 5 features output x,y

adam50Model = CoordCropModel(
    model=adam50,
    loss=torch.nn.HuberLoss(),
    opt_class=torch.optim.Adam,
    dataloaders=dataloader_dict,
    lr=0.01
    )
adam50Model.fit(100)

In [ ]:
adam50Model.get_loss_graph(), adam50Model.get_r2_graph()

## Final Model

In [ ]:
adam250Overfit = LinearRegression(5, 2, 250) # 5 features output x,y

adam250Overfit = CoordCropModel(
    model=adam250Overfit,
    loss=torch.nn.HuberLoss(),
    opt_class=torch.optim.Adam,
    dataloaders=dataloader_dict,
    lr=0.005
    )
adam250Overfit.fit(150)

In [ ]:
adam250Overfit.get_r2_graph(), adam250Overfit.get_loss_graph()

In [ ]:
min(adam250Overfit.valid_loss)

In [ ]:
adam250Overfit.valid_loss.index(min(adam250Overfit.valid_loss))

In [ ]:
adam250Final = LinearRegression(5, 2, 250) # 5 features output x,y

adam250FModel = CoordCropModel(
    model=adam250Final,
    loss=torch.nn.HuberLoss(),
    opt_class=torch.optim.Adam,
    dataloaders=dataloader_dict,
    lr=0.005
    )
adam250FModel.fit(80)

In [ ]:
adam250FModel.r2[-1]

In [ ]:
adam250FModel.get_loss_graph("for Adam-Huber-250"), adam250FModel.get_r2_graph("for Adam-Huber-250")

In [ ]:
torch.save(adam250Model.model.state_dict(), "/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/LoadRegressionAdam250.pth")

Train test

In [ ]:
adam250 = LinearRegression(5, 2, 250) # 5 features output x,y

adam250Model = CoordCropModel(
    model=adam250,
    loss=torch.nn.HuberLoss(),
    opt_class=torch.optim.Adam,
    dataloaders=dataloader_dict,
    lr=0.005
    )
adam250Model.fit(80)

In [ ]:
adam250Model.model

In [ ]:
torch.save(adam250Model.model.state_dict(), "/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/LoadRegressionAdam250.pth")